# Notebook 2 — QLoRA Fine-tuning

Run this on **Google Colab** (free T4 GPU) or Kaggle.

**Runtime:** ~2 hours for 1000 examples, 3 epochs on T4.

**Output:** `iam-policy-adapter/` (LoRA weights, ~50MB)

In [ ]:
# Run once to install dependencies
!pip install unsloth trl datasets peft accelerate bitsandbytes -q

## 1. Load base model in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/llama-3.2-3b-instruct',
    max_seq_length = 2048,
    load_in_4bit = True,
)
print('Model loaded')

## 2. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = 'none',
)
model.print_trainable_parameters()

## 3. Load and format dataset

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def format_prompt(ex):
    return (
        f"### Instruction:\n{ex['instruction']}\n\n"
        f"### Input:\n{ex['input']}\n\n"
        f"### Response:\n{json.dumps(ex['output'], indent=2)}"
    )

raw = load_jsonl('data/processed/train.jsonl')
dataset = Dataset.from_list([{'text': format_prompt(ex)} for ex in raw])
print(f'{len(dataset)} training examples')
print(dataset[0]['text'][:500])

## 4. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        output_dir = 'outputs',
        logging_steps = 10,
        save_steps = 100,
    ),
)
trainer.train()

## 5. Save adapter

In [ ]:
model.save_pretrained('iam-policy-adapter')
tokenizer.save_pretrained('iam-policy-adapter')
print('Adapter saved — upload to HuggingFace Hub next')